# BDC Statistics Explore 2026 — Final Dual-SigLIP Ensemble (Single GPU)

Self-contained final pipeline designed from the completed dataset forensics and prior experiments.

**Model A — Semantic Anchor**
- `google/siglip2-base-patch16-384`
- independent disaster-conditioned severity heads
- clean aspect-preserving preprocessing
- 2 frozen-head epochs + 4 partial fine-tuning epochs (last 4 vision blocks)

**Model B — Robust Counter-View**
- fresh copy of the same pretrained backbone
- global severity + bounded disaster-specific residual heads
- JPEG/source perturbation, random letterbox placement, component/conflict weighting
- 2 frozen-head epochs + 4 partial fine-tuning epochs
- internal snapshot blend of partial epochs 3 and 4

**Final inference**
- disaster probability: Model A
- severity probability: 50/50 Model A + Model B

The notebook trains only from labeled TRAIN and treats TEST as unlabeled inference data. No TEST-label rules, pseudo-labels, retrieval, filename-range logic, or external model probability artifacts are used.

## 1. GPU and lightweight dependencies

In [1]:
!nvidia-smi

Thu Sep  3 07:20:39 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 5090        On  |   00000000:01:00.0 Off |                  N/A |
|  0%   33C    P8              6W /  500W |    2039MiB /  32607MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# Keep the Vast.ai PyTorch/CUDA build intact.
%pip install -q -U "transformers>=4.56,<5" "huggingface_hub>=0.34" imagehash pandas pillow tqdm

Note: you may need to restart the kernel to use updated packages.


## 2. Imports, paths, and runtime policy

In [3]:
from pathlib import Path
import os, io, json, math, random, time, hashlib, gc
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
from PIL import Image, ImageOps, ImageEnhance
import imagehash
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import SiglipVisionModel, AutoImageProcessor
from huggingface_hub import snapshot_download

TRAIN_DIR = Path('/workspace/dataset/SE/TRAIN')
TEST_DIR = Path('/workspace/dataset/SE/TEST')
SOLUTION_PATH = Path('/workspace/dataset/SE/TRAIN/Solution.csv')
OUTPUT_DIR = Path('/workspace/output/final_dual_siglip_single_gpu')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

assert TRAIN_DIR.is_dir(), f'Missing TRAIN_DIR: {TRAIN_DIR}'
assert TEST_DIR.is_dir(), f'Missing TEST_DIR: {TEST_DIR}'
assert SOLUTION_PATH.is_file(), f'Missing SOLUTION_PATH: {SOLUTION_PATH}'

print('TRAIN   :', TRAIN_DIR)
print('TEST    :', TEST_DIR)
print('SOLUTION:', SOLUTION_PATH)
print('OUTPUT  :', OUTPUT_DIR)

TRAIN   : /workspace/dataset/SE/TRAIN
TEST    : /workspace/dataset/SE/TEST
SOLUTION: /workspace/dataset/SE/TRAIN/Solution.csv
OUTPUT  : /workspace/output/final_dual_siglip_single_gpu


In [4]:
assert torch.cuda.is_available(), 'A CUDA GPU is required.'
torch.cuda.set_device(0)
DEVICE = torch.device('cuda:0')
props = torch.cuda.get_device_properties(0)
VRAM_GB = props.total_memory / (1024**3)

if VRAM_GB >= 40:
    MICRO_BATCH = 32
elif VRAM_GB >= 20:
    MICRO_BATCH = 16
elif VRAM_GB >= 12:
    MICRO_BATCH = 8
else:
    MICRO_BATCH = 4

EFFECTIVE_BATCH = 32
ACCUM_STEPS = math.ceil(EFFECTIVE_BATCH / MICRO_BATCH)
EVAL_BATCH = min(64, max(16, MICRO_BATCH * 2))
NUM_WORKERS = min(12, max(4, (os.cpu_count() or 8) // 4))
USE_GRAD_CHECKPOINTING = VRAM_GB < 20
USE_BF16 = bool(torch.cuda.is_bf16_supported())
AMP_DTYPE = torch.bfloat16 if USE_BF16 else torch.float16

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.backends.cudnn.benchmark = True
torch.set_float32_matmul_precision('high')

print('torch:', torch.__version__)
print('cuda build:', torch.version.cuda)
print('GPU:', props.name)
print(f'VRAM: {VRAM_GB:.1f} GiB')
print('AMP:', 'BF16' if USE_BF16 else 'FP16')
print('micro batch:', MICRO_BATCH)
print('gradient accumulation:', ACCUM_STEPS)
print('effective batch target:', MICRO_BATCH * ACCUM_STEPS)
print('eval batch:', EVAL_BATCH)
print('workers:', NUM_WORKERS)
print('gradient checkpointing:', USE_GRAD_CHECKPOINTING)

torch: 2.11.0+cu128
cuda build: 12.8
GPU: NVIDIA GeForce RTX 5090
VRAM: 31.4 GiB
AMP: BF16
micro batch: 16
gradient accumulation: 2
effective batch target: 32
eval batch: 32
workers: 4
gradient checkpointing: False


## 3. Cache SigLIP2 once

In [5]:
MODEL_ID = 'google/siglip2-base-patch16-384'
MODEL_CACHE = snapshot_download(MODEL_ID)
print('Cached model at:', MODEL_CACHE)

Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

Cached model at: /root/.cache/huggingface/hub/models--google--siglip2-base-patch16-384/snapshots/f775b65a79762255128c981547af89addcfe0f88


## 4. Labels, TRAIN/TEST enumeration, and common image helpers

In [6]:
SEED = 20260903
IMAGE_SIZE = 384
PAD_RGB = (128,128,128)
IMAGE_EXTS = {'.jpg','.jpeg','.png','.jfif','.bmp','.webp'}

JENIS = ['BANJIR','GEMPA BUMI','KEBAKARAN']
KERUSAKAN = ['KERUSAKAN RINGAN','KERUSAKAN SEDANG','KERUSAKAN BERAT']
JENIS_TO_IDX = {x:i for i,x in enumerate(JENIS)}
KER_TO_IDX = {x:i for i,x in enumerate(KERUSAKAN)}
IDX_TO_JENIS = {i:x for i,x in enumerate(JENIS)}
IDX_TO_KER = {i:x for i,x in enumerate(KERUSAKAN)}

# Official competition submission encoding.
SUB_JENIS = {'BANJIR':1, 'GEMPA BUMI':2, 'KEBAKARAN':3}
SUB_KER = {'KERUSAKAN BERAT':1, 'KERUSAKAN RINGAN':2, 'KERUSAKAN SEDANG':3}


def seed_all(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def robust_rgb(im):
    im = ImageOps.exif_transpose(im)
    has_alpha = im.mode in ('RGBA','LA') or (im.mode == 'P' and 'transparency' in im.info)
    if has_alpha:
        rgba = im.convert('RGBA')
        bg = Image.new('RGBA', rgba.size, (128,128,128,255))
        return Image.alpha_composite(bg, rgba).convert('RGB')
    return im.convert('RGB')


def enumerate_train(root):
    rows=[]
    for jenis in JENIS:
        for ker in KERUSAKAN:
            d = root / jenis / ker
            assert d.is_dir(), f'Missing class directory: {d}'
            for p in sorted(d.iterdir()):
                if p.is_file() and p.suffix.lower() in IMAGE_EXTS:
                    ji=JENIS_TO_IDX[jenis]; ki=KER_TO_IDX[ker]
                    rows.append({
                        'path':str(p), 'jenis':jenis, 'kerusakan':ker,
                        'jenis_idx':ji, 'kerusakan_idx':ki,
                        'joint_idx':ji*3+ki,
                    })
    df=pd.DataFrame(rows)
    assert len(df)>0 and df.path.is_unique
    return df


def enumerate_test(root):
    rows=[]
    for p in root.iterdir():
        if p.is_file() and p.suffix.lower() in IMAGE_EXTS:
            rows.append({'path':str(p),'id':str(p.stem)})
    def sort_key(r):
        return (0,int(r['id'])) if r['id'].isdigit() else (1,r['id'])
    df=pd.DataFrame(sorted(rows,key=sort_key))
    assert len(df)>0 and df.id.is_unique
    return df

train_df = enumerate_train(TRAIN_DIR)
test_df = enumerate_test(TEST_DIR)
print('TRAIN images:', len(train_df))
print('TEST images :', len(test_df))
display(train_df.groupby(['jenis','kerusakan']).size().unstack(fill_value=0))

TRAIN images: 17482
TEST images : 450


kerusakan,KERUSAKAN BERAT,KERUSAKAN RINGAN,KERUSAKAN SEDANG
jenis,,,
BANJIR,1968,2018,1971
GEMPA BUMI,1623,1393,2730
KEBAKARAN,2025,1746,2008


## 5. Common SigLIP normalization and transformer-layer discovery

In [7]:
processor = AutoImageProcessor.from_pretrained(MODEL_ID, local_files_only=True)
SIGLIP_MEAN = np.array(processor.image_mean,dtype=np.float32).reshape(3,1,1)
SIGLIP_STD = np.array(processor.image_std,dtype=np.float32).reshape(3,1,1)


def normalize_canvas(canvas):
    arr=np.asarray(canvas,dtype=np.float32).transpose(2,0,1)/255.0
    arr=(arr-SIGLIP_MEAN)/SIGLIP_STD
    return torch.from_numpy(arr)


def get_siglip_layers(vision):
    candidates=[]
    core=getattr(vision,'vision_model',vision)
    candidates += [
        getattr(getattr(core,'encoder',None),'layers',None),
        getattr(getattr(vision,'encoder',None),'layers',None),
    ]
    for layers in candidates:
        if layers is not None and len(layers)>0:
            return layers
    raise AttributeError('Could not locate SigLIP vision transformer layers.')


def set_vision_partial(vision,last_n=4):
    for p in vision.parameters():
        p.requires_grad=False
    layers=get_siglip_layers(vision)
    assert len(layers)>=last_n
    for layer in layers[-last_n:]:
        for p in layer.parameters(): p.requires_grad=True
    core=getattr(vision,'vision_model',vision)
    for attr in ('post_layernorm','head'):
        module=getattr(core,attr,None)
        if module is not None:
            for p in module.parameters(): p.requires_grad=True
    return layers


def make_cosine_scheduler(opt,total_updates,warmup_ratio=0.08):
    warm=max(1,int(total_updates*warmup_ratio))
    def fn(step):
        if step<warm:
            return max(1e-8, step/warm)
        prog=(step-warm)/max(1,total_updates-warm)
        return 0.5*(1+math.cos(math.pi*min(1.0,prog)))
    return torch.optim.lr_scheduler.LambdaLR(opt,fn)

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


# Model A — V1 Semantic Anchor

## 6. Model A preprocessing, dataset, model, and loss

In [8]:
A_DROPOUT=0.15
A_HEAD_EPOCHS=2
A_PARTIAL_EPOCHS=4
A_UNFREEZE_LAST_N=4
A_HEAD_LR=7e-4
A_PARTIAL_HEAD_LR=1e-4
A_PARTIAL_BACKBONE_LR=1e-5
A_WEIGHT_DECAY=0.05
A_WARMUP_RATIO=0.08
A_GRAD_CLIP=1.0
A_W_DISASTER=0.35
A_W_SEVERITY=0.65

A_HFLIP_P=0.50
A_BRIGHTNESS=0.12
A_CONTRAST=0.12
A_SATURATION=0.08

A_CKPT = OUTPUT_DIR/'model_a_partial_epoch4.pt'
A_PROBS = OUTPUT_DIR/'model_a_test_probabilities.npz'


def decode_a(path,train=False):
    with Image.open(path) as src:
        im=robust_rgb(src)
    if train:
        if random.random()<A_HFLIP_P:
            im=ImageOps.mirror(im)
        im=ImageEnhance.Brightness(im).enhance(1+random.uniform(-A_BRIGHTNESS,A_BRIGHTNESS))
        im=ImageEnhance.Contrast(im).enhance(1+random.uniform(-A_CONTRAST,A_CONTRAST))
        im=ImageEnhance.Color(im).enhance(1+random.uniform(-A_SATURATION,A_SATURATION))
    w,h=im.size
    scale=min(IMAGE_SIZE/w,IMAGE_SIZE/h)
    nw=max(1,round(w*scale)); nh=max(1,round(h*scale))
    im=im.resize((nw,nh),Image.Resampling.BICUBIC)
    ox=(IMAGE_SIZE-nw)//2; oy=(IMAGE_SIZE-nh)//2
    canvas=Image.new('RGB',(IMAGE_SIZE,IMAGE_SIZE),PAD_RGB)
    canvas.paste(im,(ox,oy))
    return normalize_canvas(canvas)


class TrainDSA(Dataset):
    def __init__(self,df): self.df=df.reset_index(drop=True)
    def __len__(self): return len(self.df)
    def __getitem__(self,i):
        r=self.df.iloc[i]
        return {
            'pixel_values':decode_a(r.path,True),
            'jenis_idx':int(r.jenis_idx),
            'kerusakan_idx':int(r.kerusakan_idx),
        }


class TestDSA(Dataset):
    def __init__(self,df): self.df=df.reset_index(drop=True)
    def __len__(self): return len(self.df)
    def __getitem__(self,i):
        return {'pixel_values':decode_a(self.df.iloc[i].path,False)}


class SemanticAnchor(nn.Module):
    def __init__(self):
        super().__init__()
        self.vision=SiglipVisionModel.from_pretrained(MODEL_ID,local_files_only=True)
        if USE_GRAD_CHECKPOINTING and hasattr(self.vision,'gradient_checkpointing_enable'):
            self.vision.gradient_checkpointing_enable()
        d=self.vision.config.hidden_size
        self.feature_norm=nn.LayerNorm(d)
        self.drop=nn.Dropout(A_DROPOUT)
        self.disaster=nn.Linear(d,3)
        self.severity=nn.ModuleList([nn.Linear(d,3) for _ in range(3)])

    def features(self,x):
        out=self.vision(pixel_values=x,return_dict=True)
        z=out.pooler_output if getattr(out,'pooler_output',None) is not None else out.last_hidden_state.mean(1)
        return self.drop(self.feature_norm(z.float()))

    def forward(self,x):
        z=self.features(x)
        j=self.disaster(z)
        cond=torch.stack([h(z) for h in self.severity],dim=1)
        jp=F.softmax(j.float(),dim=-1)
        cp=F.softmax(cond.float(),dim=-1)
        kp=torch.sum(jp.unsqueeze(-1)*cp,dim=1)
        return j,cond,kp


def set_stage_a(model,stage):
    for p in model.vision.parameters(): p.requires_grad=False
    for module in [model.feature_norm,model.disaster,model.severity]:
        for p in module.parameters(): p.requires_grad=True
    if stage=='partial':
        set_vision_partial(model.vision,A_UNFREEZE_LAST_N)


def loss_a(j,cond,yj,yk):
    lj=F.cross_entropy(j,yj)
    true_cond=cond[torch.arange(len(yj),device=yj.device),yj]
    lk=F.cross_entropy(true_cond,yk)
    return A_W_DISASTER*lj + A_W_SEVERITY*lk


def optimizer_a(model,stage):
    head=[]; backbone=[]
    for name,p in model.named_parameters():
        if not p.requires_grad: continue
        (backbone if name.startswith('vision.') else head).append(p)
    groups=[]
    if backbone: groups.append({'params':backbone,'lr':A_PARTIAL_BACKBONE_LR})
    if head: groups.append({'params':head,'lr':A_HEAD_LR if stage=='heads' else A_PARTIAL_HEAD_LR})
    return torch.optim.AdamW(groups,weight_decay=A_WEIGHT_DECAY)

## 7. Model A smoke test

In [9]:
seed_all(SEED)
_model=SemanticAnchor().to(DEVICE).eval()
_x=torch.zeros(1,3,IMAGE_SIZE,IMAGE_SIZE,device=DEVICE)
with torch.no_grad(),torch.autocast('cuda',dtype=AMP_DTYPE):
    _j,_c,_k=_model(_x)
assert _j.shape==(1,3) and _c.shape==(1,3,3) and _k.shape==(1,3)
_layers=get_siglip_layers(_model.vision)
assert len(_layers)>=A_UNFREEZE_LAST_N
set_stage_a(_model,'partial')
assert all(any(p.requires_grad for p in layer.parameters()) for layer in _layers[-A_UNFREEZE_LAST_N:])
print('MODEL A SMOKE OK | feature dim:',_model.disaster.in_features,'| blocks:',len(_layers))
del _model,_x,_j,_c,_k; gc.collect(); torch.cuda.empty_cache()

MODEL A SMOKE OK | feature dim: 768 | blocks: 12


## 8. Train Model A on 100% TRAIN and infer TEST

In [10]:
def train_epoch_a(model,loader,opt,sched,scaler):
    model.train(); opt.zero_grad(set_to_none=True)
    total=0.0; n=0
    pbar=tqdm(enumerate(loader),total=len(loader),leave=False)
    for step,b in pbar:
        x=b['pixel_values'].to(DEVICE,non_blocking=True)
        yj=b['jenis_idx'].to(DEVICE,non_blocking=True)
        yk=b['kerusakan_idx'].to(DEVICE,non_blocking=True)
        with torch.autocast('cuda',dtype=AMP_DTYPE):
            j,c,_=model(x)
            loss=loss_a(j,c,yj,yk)
            bl=loss/ACCUM_STEPS
        if scaler.is_enabled(): scaler.scale(bl).backward()
        else: bl.backward()
        if ((step+1)%ACCUM_STEPS==0) or (step+1==len(loader)):
            if scaler.is_enabled():
                scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(),A_GRAD_CLIP)
                scaler.step(opt); scaler.update()
            else:
                torch.nn.utils.clip_grad_norm_(model.parameters(),A_GRAD_CLIP); opt.step()
            opt.zero_grad(set_to_none=True); sched.step()
        total += float(loss.detach())*len(x); n += len(x)
        pbar.set_postfix(loss=f'{total/max(1,n):.4f}')
    return total/max(1,n)


@torch.no_grad()
def predict_a(model,loader):
    model.eval(); jp=[]; kp=[]
    for b in tqdm(loader,desc='Model A TEST',leave=False):
        x=b['pixel_values'].to(DEVICE,non_blocking=True)
        with torch.autocast('cuda',dtype=AMP_DTYPE):
            j,c,_=model(x)
        # recompute routing in FP32
        jprob=F.softmax(j.float(),dim=-1)
        cprob=F.softmax(c.float(),dim=-1)
        kprob=torch.sum(jprob.unsqueeze(-1)*cprob,dim=1)
        jp.append(jprob.cpu().numpy()); kp.append(kprob.cpu().numpy())
    jp=np.concatenate(jp).astype(np.float32); kp=np.concatenate(kp).astype(np.float32)
    jp/=np.clip(jp.sum(1,keepdims=True),1e-12,None)
    kp/=np.clip(kp.sum(1,keepdims=True),1e-12,None)
    return jp,kp


seed_all(SEED)
train_loader_a=DataLoader(
    TrainDSA(train_df),batch_size=MICRO_BATCH,shuffle=True,num_workers=NUM_WORKERS,
    pin_memory=True,persistent_workers=(NUM_WORKERS>0),drop_last=False,
    generator=torch.Generator().manual_seed(SEED),
)
test_loader_a=DataLoader(
    TestDSA(test_df),batch_size=EVAL_BATCH,shuffle=False,num_workers=NUM_WORKERS,
    pin_memory=True,persistent_workers=(NUM_WORKERS>0),
)

model_a=SemanticAnchor().to(DEVICE)
scaler=torch.amp.GradScaler('cuda',enabled=not USE_BF16)
history_a=[]
updates_per_epoch=math.ceil(len(train_loader_a)/ACCUM_STEPS)

set_stage_a(model_a,'heads')
opt=optimizer_a(model_a,'heads')
sched=make_cosine_scheduler(opt,updates_per_epoch*A_HEAD_EPOCHS,A_WARMUP_RATIO)
for ep in range(1,A_HEAD_EPOCHS+1):
    t=time.time(); loss=train_epoch_a(model_a,train_loader_a,opt,sched,scaler)
    rec={'stage':'heads','epoch':ep,'loss':loss,'minutes':(time.time()-t)/60}
    history_a.append(rec); print('A',rec)

set_stage_a(model_a,'partial')
opt=optimizer_a(model_a,'partial')
sched=make_cosine_scheduler(opt,updates_per_epoch*A_PARTIAL_EPOCHS,A_WARMUP_RATIO)
for ep in range(1,A_PARTIAL_EPOCHS+1):
    t=time.time(); loss=train_epoch_a(model_a,train_loader_a,opt,sched,scaler)
    rec={'stage':'partial','epoch':ep,'loss':loss,'minutes':(time.time()-t)/60}
    history_a.append(rec); print('A',rec)
    if ep==A_PARTIAL_EPOCHS:
        torch.save({'model':model_a.state_dict(),'history':history_a},A_CKPT)

(OUTPUT_DIR/'model_a_history.json').write_text(json.dumps(history_a,indent=2))
a_j,a_k=predict_a(model_a,test_loader_a)
np.savez_compressed(A_PROBS,ids=test_df.id.astype(str).to_numpy(),jenis_prob=a_j,kerusakan_prob=a_k)
print('Saved Model A:',A_CKPT)
print('Saved Model A probabilities:',A_PROBS)

del model_a,opt,sched,scaler,train_loader_a,test_loader_a
gc.collect(); torch.cuda.empty_cache()

  0%|          | 0/1093 [00:00<?, ?it/s]

A {'stage': 'heads', 'epoch': 1, 'loss': 0.4341929174167806, 'minutes': 1.8273107449213664}


  0%|          | 0/1093 [00:00<?, ?it/s]

A {'stage': 'heads', 'epoch': 2, 'loss': 0.31753986091625225, 'minutes': 1.7846899271011352}


  0%|          | 0/1093 [00:00<?, ?it/s]

...

A {'stage': 'partial', 'epoch': 3, 'loss': 0.12189285623645049, 'minutes': 1.7351242502530415}


  0%|          | 0/1093 [00:00<?, ?it/s]

A {'stage': 'partial', 'epoch': 4, 'loss': 0.08750204268142713, 'minutes': 1.7461950103441874}


Model A TEST:   0%|          | 0/15 [00:00<?, ?it/s]

Saved Model A: /workspace/output/final_dual_siglip_single_gpu/model_a_partial_epoch4.pt
Saved Model A probabilities: /workspace/output/final_dual_siglip_single_gpu/model_a_test_probabilities.npz


# Model B — V2 Robust Counter-View

## 9. TRAIN-only exact + strict-near component manifest

In [11]:
HASH_CACHE = OUTPUT_DIR/'train_hashes.csv'
B_MANIFEST_PATH = OUTPUT_DIR/'model_b_weighted_manifest.csv'


def hash_one(path):
    p=Path(path)
    sha=hashlib.sha256(p.read_bytes()).hexdigest()
    with Image.open(p) as im:
        im=robust_rgb(im)
        return sha,str(imagehash.phash(im)),str(imagehash.dhash(im)),str(imagehash.average_hash(im))


if HASH_CACHE.exists():
    hdf=pd.read_csv(HASH_CACHE,dtype=str)
    print('Loaded hash cache:',HASH_CACHE)
else:
    workers=min(32,os.cpu_count() or 8)
    with ThreadPoolExecutor(max_workers=workers) as ex:
        vals=list(tqdm(ex.map(hash_one,train_df.path),total=len(train_df),desc='Hashing TRAIN'))
    hdf=pd.DataFrame(vals,columns=['sha256','phash','dhash','ahash'])
    hdf.insert(0,'path',train_df.path.values)
    hdf.to_csv(HASH_CACHE,index=False)
    print('Saved hash cache:',HASH_CACHE)
assert list(hdf.path)==list(train_df.path)


class DSU:
    def __init__(self,n): self.p=list(range(n)); self.r=[0]*n
    def find(self,x):
        while self.p[x]!=x:
            self.p[x]=self.p[self.p[x]]; x=self.p[x]
        return x
    def union(self,a,b):
        a,b=self.find(a),self.find(b)
        if a==b:return
        if self.r[a]<self.r[b]:a,b=b,a
        self.p[b]=a
        if self.r[a]==self.r[b]:self.r[a]+=1


def ham(a,b): return (int(a)^int(b)).bit_count()


def build_groups(hdf):
    n=len(hdf); dsu=DSU(n)
    sha_map={}
    for i,s in enumerate(hdf.sha256):
        if s in sha_map: dsu.union(i,sha_map[s])
        else: sha_map[s]=i
    ph=np.array([int(x,16) for x in hdf.phash],dtype=object)
    dh=np.array([int(x,16) for x in hdf.dhash],dtype=object)
    ah=np.array([int(x,16) for x in hdf.ahash],dtype=object)
    buckets={}; candidates=set()
    for i,v0 in enumerate(ph):
        v=int(v0)
        for band in range(4):
            key=(band,(v>>(16*band))&0xFFFF)
            for j in buckets.get(key,[]): candidates.add((j,i))
            buckets.setdefault(key,[]).append(i)
    kept=0
    for a,b in tqdm(candidates,desc='Strict near-duplicate edges'):
        if ham(ph[a],ph[b])<=1 and ham(dh[a],dh[b])<=2 and ham(ah[a],ah[b])<=4:
            dsu.union(a,b); kept+=1
    roots=[dsu.find(i) for i in range(n)]
    remap={r:j for j,r in enumerate(sorted(set(roots)))}
    return np.array([remap[r] for r in roots],dtype=np.int64),kept


if B_MANIFEST_PATH.exists():
    manifest_b=pd.read_csv(B_MANIFEST_PATH)
    print('Loaded Model B manifest:',B_MANIFEST_PATH)
else:
    scene_group,near_edges=build_groups(hdf)
    manifest_b=train_df.copy(); manifest_b['scene_group']=scene_group
    manifest_b=manifest_b.join(manifest_b.groupby('scene_group').size().rename('group_size'),on='scene_group')
    manifest_b=manifest_b.join(manifest_b.groupby('scene_group').kerusakan_idx.nunique().rename('severity_nlabels'),on='scene_group')
    manifest_b=manifest_b.join(manifest_b.groupby('scene_group').jenis_idx.nunique().rename('disaster_nlabels'),on='scene_group')
    manifest_b['severity_conflict']=manifest_b.severity_nlabels.gt(1)
    manifest_b['disaster_conflict']=manifest_b.disaster_nlabels.gt(1)
    manifest_b['component_weight_raw']=1.0/np.sqrt(manifest_b.group_size.astype(float))
    denom=manifest_b.groupby('joint_idx').component_weight_raw.transform('mean')
    manifest_b['component_weight']=manifest_b.component_weight_raw/denom
    manifest_b['component_weight']=manifest_b.component_weight.clip(0.20,2.50)
    manifest_b['component_weight']/=manifest_b.groupby('joint_idx').component_weight.transform('mean')
    manifest_b['severity_weight']=manifest_b.component_weight*np.where(manifest_b.severity_conflict,0.25,1.0)
    manifest_b['disaster_weight']=manifest_b.component_weight*np.where(manifest_b.disaster_conflict,0.25,1.0)
    manifest_b.to_csv(B_MANIFEST_PATH,index=False)
    print('strict near edges:',near_edges)
    print('Saved Model B manifest:',B_MANIFEST_PATH)

assert len(manifest_b)==len(train_df) and manifest_b.path.is_unique
print('scene groups:',manifest_b.scene_group.nunique())
print('non-singleton images:',int((manifest_b.group_size>1).sum()))
print('severity-conflict images:',int(manifest_b.severity_conflict.sum()))
print('disaster-conflict images:',int(manifest_b.disaster_conflict.sum()))

Hashing TRAIN:   0%|          | 0/17482 [00:00<?, ?it/s]

Saved hash cache: /workspace/output/final_dual_siglip_single_gpu/train_hashes.csv


Strict near-duplicate edges:   0%|          | 0/152663 [00:00<?, ?it/s]

strict near edges: 4111
Saved Model B manifest: /workspace/output/final_dual_siglip_single_gpu/model_b_weighted_manifest.csv
scene groups: 15296
non-singleton images: 3701
severity-conflict images: 376
disaster-conflict images: 2


## 10. Model B preprocessing, model, and loss

In [12]:
B_DROPOUT=0.15
B_HEAD_EPOCHS=2
B_PARTIAL_EPOCHS=4
B_UNFREEZE_LAST_N=4
B_HEAD_LR=7e-4
B_PARTIAL_HEAD_LR=1e-4
B_PARTIAL_BACKBONE_LR=1e-5
B_WEIGHT_DECAY=0.05
B_WARMUP_RATIO=0.08
B_GRAD_CLIP=1.0

B_W_DISASTER=0.30
B_W_COND_SEVERITY=0.55
B_W_GLOBAL_SEVERITY=0.15
B_LABEL_SMOOTHING=0.05
B_RESIDUAL_ALPHA=0.50

B_HFLIP_P=0.50
B_BRIGHTNESS=0.10
B_CONTRAST=0.10
B_SATURATION=0.07
B_JPEG_P=0.35
B_JPEG_Q_MIN=70
B_JPEG_Q_MAX=95

B_SNAP3=OUTPUT_DIR/'model_b_partial_epoch3.pt'
B_SNAP4=OUTPUT_DIR/'model_b_partial_epoch4.pt'
B_PROBS=OUTPUT_DIR/'model_b_test_probabilities_snapshot34.npz'


def jpeg_perturb(im):
    q=random.randint(B_JPEG_Q_MIN,B_JPEG_Q_MAX)
    buf=io.BytesIO(); im.save(buf,format='JPEG',quality=q,subsampling=2); buf.seek(0)
    with Image.open(buf) as rec: return rec.convert('RGB').copy()


def decode_b(path,train=False):
    with Image.open(path) as src: im=robust_rgb(src)
    if train:
        if random.random()<B_HFLIP_P: im=ImageOps.mirror(im)
        if random.random()<B_JPEG_P: im=jpeg_perturb(im)
        im=ImageEnhance.Brightness(im).enhance(1+random.uniform(-B_BRIGHTNESS,B_BRIGHTNESS))
        im=ImageEnhance.Contrast(im).enhance(1+random.uniform(-B_CONTRAST,B_CONTRAST))
        im=ImageEnhance.Color(im).enhance(1+random.uniform(-B_SATURATION,B_SATURATION))
    w,h=im.size
    scale=min(IMAGE_SIZE/w,IMAGE_SIZE/h)
    nw=max(1,round(w*scale)); nh=max(1,round(h*scale))
    im=im.resize((nw,nh),Image.Resampling.BICUBIC)
    free_x=IMAGE_SIZE-nw; free_y=IMAGE_SIZE-nh
    if train:
        ox=random.randint(0,free_x) if free_x>0 else 0
        oy=random.randint(0,free_y) if free_y>0 else 0
    else:
        ox=free_x//2; oy=free_y//2
    canvas=Image.new('RGB',(IMAGE_SIZE,IMAGE_SIZE),PAD_RGB); canvas.paste(im,(ox,oy))
    return normalize_canvas(canvas)


class TrainDSB(Dataset):
    def __init__(self,df): self.df=df.reset_index(drop=True)
    def __len__(self): return len(self.df)
    def __getitem__(self,i):
        r=self.df.iloc[i]
        return {
            'pixel_values':decode_b(r.path,True),
            'jenis_idx':int(r.jenis_idx),'kerusakan_idx':int(r.kerusakan_idx),
            'disaster_weight':float(r.disaster_weight),'severity_weight':float(r.severity_weight),
        }


class TestDSB(Dataset):
    def __init__(self,df): self.df=df.reset_index(drop=True)
    def __len__(self): return len(self.df)
    def __getitem__(self,i): return {'pixel_values':decode_b(self.df.iloc[i].path,False)}


class RobustCounterView(nn.Module):
    def __init__(self):
        super().__init__()
        self.vision=SiglipVisionModel.from_pretrained(MODEL_ID,local_files_only=True)
        if USE_GRAD_CHECKPOINTING and hasattr(self.vision,'gradient_checkpointing_enable'):
            self.vision.gradient_checkpointing_enable()
        d=self.vision.config.hidden_size
        self.feature_norm=nn.LayerNorm(d); self.drop=nn.Dropout(B_DROPOUT)
        self.disaster=nn.Linear(d,3); self.global_severity=nn.Linear(d,3)
        self.severity_residual=nn.ModuleList([nn.Linear(d,3) for _ in range(3)])

    def features(self,x):
        out=self.vision(pixel_values=x,return_dict=True)
        z=out.pooler_output if getattr(out,'pooler_output',None) is not None else out.last_hidden_state.mean(1)
        return self.drop(self.feature_norm(z.float()))

    def forward(self,x):
        z=self.features(x)
        j=self.disaster(z); g=self.global_severity(z)
        residual=torch.stack([torch.tanh(h(z)) for h in self.severity_residual],dim=1)
        cond=g.unsqueeze(1)+B_RESIDUAL_ALPHA*residual
        jp=F.softmax(j.float(),dim=-1); cp=F.softmax(cond.float(),dim=-1)
        kp=torch.sum(jp.unsqueeze(-1)*cp,dim=1)
        return j,g,cond,kp


def set_stage_b(model,stage):
    for p in model.vision.parameters(): p.requires_grad=False
    for module in [model.feature_norm,model.disaster,model.global_severity,model.severity_residual]:
        for p in module.parameters(): p.requires_grad=True
    if stage=='partial': set_vision_partial(model.vision,B_UNFREEZE_LAST_N)


def weighted_mean(x,w): return (x*w).sum()/w.sum().clamp_min(1e-6)


def loss_b(j,g,cond,yj,yk,wd,wk):
    lj=F.cross_entropy(j,yj,reduction='none')
    lg=F.cross_entropy(g,yk,reduction='none',label_smoothing=B_LABEL_SMOOTHING)
    true_cond=cond[torch.arange(len(yj),device=yj.device),yj]
    lc=F.cross_entropy(true_cond,yk,reduction='none',label_smoothing=B_LABEL_SMOOTHING)
    return B_W_DISASTER*weighted_mean(lj,wd)+B_W_COND_SEVERITY*weighted_mean(lc,wk)+B_W_GLOBAL_SEVERITY*weighted_mean(lg,wk)


def optimizer_b(model,stage):
    head=[]; backbone=[]
    for name,p in model.named_parameters():
        if not p.requires_grad: continue
        (backbone if name.startswith('vision.') else head).append(p)
    groups=[]
    if backbone: groups.append({'params':backbone,'lr':B_PARTIAL_BACKBONE_LR})
    if head: groups.append({'params':head,'lr':B_HEAD_LR if stage=='heads' else B_PARTIAL_HEAD_LR})
    return torch.optim.AdamW(groups,weight_decay=B_WEIGHT_DECAY)

## 11. Model B smoke test

In [13]:
seed_all(SEED+101)
_model=RobustCounterView().to(DEVICE).eval(); _x=torch.zeros(1,3,IMAGE_SIZE,IMAGE_SIZE,device=DEVICE)
with torch.no_grad(),torch.autocast('cuda',dtype=AMP_DTYPE): _j,_g,_c,_k=_model(_x)
assert _j.shape==(1,3) and _g.shape==(1,3) and _c.shape==(1,3,3) and _k.shape==(1,3)
_layers=get_siglip_layers(_model.vision); assert len(_layers)>=B_UNFREEZE_LAST_N
set_stage_b(_model,'partial')
assert all(any(p.requires_grad for p in layer.parameters()) for layer in _layers[-B_UNFREEZE_LAST_N:])
print('MODEL B SMOKE OK | feature dim:',_model.disaster.in_features,'| blocks:',len(_layers))
del _model,_x,_j,_g,_c,_k; gc.collect(); torch.cuda.empty_cache()

MODEL B SMOKE OK | feature dim: 768 | blocks: 12


## 12. Train Model B on 100% TRAIN

In [ ]:
def train_epoch_b(model,loader,opt,sched,scaler):
    model.train(); opt.zero_grad(set_to_none=True); total=0.0; n=0
    pbar=tqdm(enumerate(loader),total=len(loader),leave=False)
    for step,b in pbar:
        x=b['pixel_values'].to(DEVICE,non_blocking=True)
        yj=b['jenis_idx'].to(DEVICE,non_blocking=True); yk=b['kerusakan_idx'].to(DEVICE,non_blocking=True)
        wd=b['disaster_weight'].to(DEVICE,dtype=torch.float32,non_blocking=True)
        wk=b['severity_weight'].to(DEVICE,dtype=torch.float32,non_blocking=True)
        with torch.autocast('cuda',dtype=AMP_DTYPE):
            j,g,c,_=model(x); loss=loss_b(j,g,c,yj,yk,wd,wk); bl=loss/ACCUM_STEPS
        if scaler.is_enabled(): scaler.scale(bl).backward()
        else: bl.backward()
        if ((step+1)%ACCUM_STEPS==0) or (step+1==len(loader)):
            if scaler.is_enabled():
                scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(),B_GRAD_CLIP)
                scaler.step(opt); scaler.update()
            else:
                torch.nn.utils.clip_grad_norm_(model.parameters(),B_GRAD_CLIP); opt.step()
            opt.zero_grad(set_to_none=True); sched.step()
        total+=float(loss.detach())*len(x); n+=len(x); pbar.set_postfix(loss=f'{total/max(1,n):.4f}')
    return total/max(1,n)


seed_all(SEED+101)
train_loader_b=DataLoader(
    TrainDSB(manifest_b),batch_size=MICRO_BATCH,shuffle=True,num_workers=NUM_WORKERS,
    pin_memory=True,persistent_workers=(NUM_WORKERS>0),drop_last=False,
    generator=torch.Generator().manual_seed(SEED+101),
)
model_b=RobustCounterView().to(DEVICE)
scaler=torch.amp.GradScaler('cuda',enabled=not USE_BF16)
history_b=[]; updates_per_epoch=math.ceil(len(train_loader_b)/ACCUM_STEPS)

set_stage_b(model_b,'heads'); opt=optimizer_b(model_b,'heads')
sched=make_cosine_scheduler(opt,updates_per_epoch*B_HEAD_EPOCHS,B_WARMUP_RATIO)
for ep in range(1,B_HEAD_EPOCHS+1):
    t=time.time(); loss=train_epoch_b(model_b,train_loader_b,opt,sched,scaler)
    rec={'stage':'heads','epoch':ep,'loss':loss,'minutes':(time.time()-t)/60}; history_b.append(rec); print('B',rec)

set_stage_b(model_b,'partial'); opt=optimizer_b(model_b,'partial')
sched=make_cosine_scheduler(opt,updates_per_epoch*B_PARTIAL_EPOCHS,B_WARMUP_RATIO)
for ep in range(1,B_PARTIAL_EPOCHS+1):
    t=time.time(); loss=train_epoch_b(model_b,train_loader_b,opt,sched,scaler)
    rec={'stage':'partial','epoch':ep,'loss':loss,'minutes':(time.time()-t)/60}; history_b.append(rec); print('B',rec)
    if ep==3: torch.save({'model':model_b.state_dict(),'history':history_b},B_SNAP3)
    if ep==4: torch.save({'model':model_b.state_dict(),'history':history_b},B_SNAP4)

(OUTPUT_DIR/'model_b_history.json').write_text(json.dumps(history_b,indent=2))
print('Saved:',B_SNAP3); print('Saved:',B_SNAP4)
del model_b,opt,sched,scaler,train_loader_b; gc.collect(); torch.cuda.empty_cache()

  0%|          | 0/1093 [00:00<?, ?it/s]

B {'stage': 'heads', 'epoch': 1, 'loss': 0.528865733489389, 'minutes': 1.9595035115877788}


  0%|          | 0/1093 [00:00<?, ?it/s]

B {'stage': 'heads', 'epoch': 2, 'loss': 0.4224440895328668, 'minutes': 1.9274999976158143}


  0%|          | 0/1093 [00:00<?, ?it/s]

...

  0%|          | 0/1093 [00:00<?, ?it/s]

B {'stage': 'partial', 'epoch': 2, 'loss': 0.3123820582382358, 'minutes': 1.90264759461085}


  0%|          | 0/1093 [00:00<?, ?it/s]

B {'stage': 'partial', 'epoch': 3, 'loss': 0.25687273526379917, 'minutes': 1.8758060852686564}


  0%|          | 0/1093 [00:00<?, ?it/s]

## 13. Model B snapshot inference and internal blend

In [ ]:
@torch.no_grad()
def predict_b_snapshot(ckpt_path,test_loader):
    model=RobustCounterView().to(DEVICE)
    state=torch.load(ckpt_path,map_location='cpu'); model.load_state_dict(state['model'],strict=True); model.eval()
    jp=[]; kp=[]
    for b in tqdm(test_loader,desc=f'Infer {Path(ckpt_path).name}',leave=False):
        x=b['pixel_values'].to(DEVICE,non_blocking=True)
        with torch.autocast('cuda',dtype=AMP_DTYPE): j,g,c,_=model(x)
        jprob=F.softmax(j.float(),dim=-1); cprob=F.softmax(c.float(),dim=-1)
        kprob=torch.sum(jprob.unsqueeze(-1)*cprob,dim=1)
        jp.append(jprob.cpu().numpy()); kp.append(kprob.cpu().numpy())
    del model; gc.collect(); torch.cuda.empty_cache()
    jp=np.concatenate(jp).astype(np.float32); kp=np.concatenate(kp).astype(np.float32)
    jp/=np.clip(jp.sum(1,keepdims=True),1e-12,None); kp/=np.clip(kp.sum(1,keepdims=True),1e-12,None)
    return jp,kp


test_loader_b=DataLoader(
    TestDSB(test_df),batch_size=EVAL_BATCH,shuffle=False,num_workers=NUM_WORKERS,
    pin_memory=True,persistent_workers=(NUM_WORKERS>0),
)
b_j3,b_k3=predict_b_snapshot(B_SNAP3,test_loader_b)
b_j4,b_k4=predict_b_snapshot(B_SNAP4,test_loader_b)
b_j=(b_j3+b_j4)/2.0; b_k=(b_k3+b_k4)/2.0
b_j/=np.clip(b_j.sum(1,keepdims=True),1e-12,None); b_k/=np.clip(b_k.sum(1,keepdims=True),1e-12,None)
np.savez_compressed(B_PROBS,ids=test_df.id.astype(str).to_numpy(),jenis_prob=b_j,kerusakan_prob=b_k,
                    jenis_prob_epoch3=b_j3,kerusakan_prob_epoch3=b_k3,
                    jenis_prob_epoch4=b_j4,kerusakan_prob_epoch4=b_k4)
print('Saved Model B probabilities:',B_PROBS)
del test_loader_b; gc.collect(); torch.cuda.empty_cache()

# Final ensemble and submission

## 14. Fixed late ensemble

In [ ]:
# Fixed before TEST inference: Model A anchors disaster; A/B equal-average severity.
assert a_j.shape==b_j.shape==(len(test_df),3)
assert a_k.shape==b_k.shape==(len(test_df),3)

final_jenis_prob=a_j.copy()
final_ker_prob=0.5*a_k+0.5*b_k
final_jenis_prob/=np.clip(final_jenis_prob.sum(1,keepdims=True),1e-12,None)
final_ker_prob/=np.clip(final_ker_prob.sum(1,keepdims=True),1e-12,None)

assert np.isfinite(final_jenis_prob).all() and np.isfinite(final_ker_prob).all()
assert np.allclose(final_jenis_prob.sum(1),1,atol=1e-6)
assert np.allclose(final_ker_prob.sum(1),1,atol=1e-6)

np.savez_compressed(
    OUTPUT_DIR/'final_test_probabilities.npz',
    ids=test_df.id.astype(str).to_numpy(),
    jenis_prob=final_jenis_prob,kerusakan_prob=final_ker_prob,
    model_a_jenis_prob=a_j,model_a_kerusakan_prob=a_k,
    model_b_jenis_prob=b_j,model_b_kerusakan_prob=b_k,
)
print('Final probability tensors:',final_jenis_prob.shape,final_ker_prob.shape)

## 15. Build official numeric submission

In [ ]:
pred_j=final_jenis_prob.argmax(1)
pred_k=final_ker_prob.argmax(1)

by_id={
    str(test_df.iloc[i].id):(
        SUB_JENIS[IDX_TO_JENIS[int(pred_j[i])]],
        SUB_KER[IDX_TO_KER[int(pred_k[i])]],
    )
    for i in range(len(test_df))
}

solution=pd.read_csv(SOLUTION_PATH,sep=None,engine='python')
assert 'ID' in solution.columns and 'Target' in solution.columns
submission=solution.copy(); targets=[]
for rid in submission['ID'].astype(str):
    base,suffix=rid.rsplit('_',1)
    assert base in by_id,f'Missing TEST ID: {base}'
    if suffix=='jenis': targets.append(by_id[base][0])
    elif suffix=='kerusakan': targets.append(by_id[base][1])
    else: raise ValueError(f'Unexpected Solution.csv ID suffix: {rid}')
submission['Target']=targets

SUBMISSION_PATH=OUTPUT_DIR/'submission.csv'
submission.to_csv(SUBMISSION_PATH,index=False)
print(submission.head(12).to_string(index=False))
print('rows:',len(submission))
print('Target counts:')
print(submission.Target.value_counts().sort_index().to_string())
print('Saved:',SUBMISSION_PATH)

## 16. Final sanity checks

In [ ]:
check=pd.read_csv(SUBMISSION_PATH)
assert len(test_df)==450, f'Expected 450 TEST images, got {len(test_df)}'
assert len(check)==900, f'Expected 900 submission rows, got {len(check)}'
assert check.ID.astype(str).is_unique
assert check.Target.notna().all()
assert set(check.Target.astype(int).unique()).issubset({1,2,3})

# Exactly one jenis and one kerusakan row per TEST ID.
parts=check.ID.astype(str).str.rsplit('_',n=1,expand=True)
assert set(parts[1].unique())=={'jenis','kerusakan'}
assert parts[0].value_counts().eq(2).all()

print('FINAL SANITY OK')
print('Submission:',SUBMISSION_PATH)
print('Model A checkpoint:',A_CKPT)
print('Model B snapshots:',B_SNAP3,B_SNAP4)
print('Final probabilities:',OUTPUT_DIR/'final_test_probabilities.npz')

## Artifacts

Primary artifact to submit:

`/workspace/output/final_dual_siglip_single_gpu/submission.csv`

Additional reproducibility artifacts are stored in the same output directory: Model A checkpoint/history/probabilities, Model B snapshots/history/manifest/probabilities, and final ensemble probabilities.